# 🩺 Diabetic Retinopathy Detection — Deep Learning Clinical Training Pipeline
### **चक्षुस् (Chakshus) AI Tele-Ophthalmology Platform**
**Dataset:** APTOS 2019 Blindness Detection (Aravind Eye Hospital, India)

This notebook trains a deep convolutional neural network (ResNet Transfer Learning) on authentic clinical retinal fundus imagery to diagnose Diabetic Retinopathy severity across the 5 standard International Clinical Diabetic Retinopathy (ICDR) grades:
- **Grade 0 (No DR)**: Clean healthy retina, no microaneurysms
- **Grade 1 (Mild NPDR)**: Microaneurysms only
- **Grade 2 (Moderate NPDR)**: Microaneurysms, hard exudates, or cotton-wool patches
- **Grade 3 (Severe NPDR)**: 4:2:1 clinical rule (extensive hemorrhages, venous beading, IRMA)
- **Grade 4 (Proliferative DR)**: Neovascularization, vitreous hemorrhage, high blindness risk

> **GPU Recommended**: Go to `Runtime` -> `Change runtime type` -> Select `T4 GPU`.

In [ ]:
# 1. Environment Verification & Dependencies
!pip install -q torch torchvision torchaudio scikit-learn matplotlib pandas pillow

import os
import time
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, classification_report

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 PyTorch Version: {torch.__version__}")
print(f"🎯 Execution Device: {device}")
if torch.cuda.is_available():
    print(f"⚡ GPU Model: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Dataset Path Configuration (APTOS 2019)
DATASET_DIR = "dataset"
TRAIN_CSV = os.path.join(DATASET_DIR, "train_1.csv")
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train_images")
VAL_CSV = os.path.join(DATASET_DIR, "valid.csv")
VAL_IMG_DIR = os.path.join(DATASET_DIR, "val_images")

print(f"Checking dataset files in '{DATASET_DIR}'...")
print(f"  Train CSV exists: {os.path.exists(TRAIN_CSV)}")
print(f"  Train Imgs exist: {os.path.exists(TRAIN_IMG_DIR)}")
print(f"  Val CSV exists  : {os.path.exists(VAL_CSV)}")
print(f"  Val Imgs exist  : {os.path.exists(VAL_IMG_DIR)}")

# Read dataframes
df_train = pd.read_csv(TRAIN_CSV)
df_val = pd.read_csv(VAL_CSV)
print(f"\n✅ Total Training Scans   : {len(df_train)}")
print(f"✅ Total Validation Scans : {len(df_val)}")

In [ ]:
# 3. Exploratory Data Analysis (EDA) — Class Distribution
CLASS_NAMES = ["0: No DR", "1: Mild", "2: Moderate", "3: Severe", "4: Proliferative"]
train_counts = df_train["diagnosis"].value_counts().sort_index()

plt.figure(figsize=(9, 4.5))
bars = plt.bar(CLASS_NAMES, train_counts, color=["#10B981", "#3B82F6", "#F59E0B", "#EF4444", "#7F1D1D"], edgecolor="#1E293B", lw=1.2)
plt.title("APTOS 2019 Diabetic Retinopathy Class Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Clinical ICDR Severity Stage", fontweight="bold")
plt.ylabel("Number of Retinal Photographs", fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 20, f"{yval:,}", ha="center", va="bottom", fontweight="bold", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Clinical Retinal Preprocessing Pipeline (Ben Graham Method & Circular Crop)
def crop_image_from_gray(img: Image.Image, tol: int = 7) -> Image.Image:
    """Crops unilluminated black margins around the retinal circular aperture."""
    arr = np.array(img)
    gray = np.mean(arr, axis=-1)
    mask = gray > tol
    if not mask.any():
        return img
    rows = mask.any(axis=1)
    cols = mask.any(axis=0)
    cropped_arr = arr[np.ix_(rows, cols)]
    if cropped_arr.shape[0] == 0 or cropped_arr.shape[1] == 0:
        return img
    return Image.fromarray(cropped_arr)

def circle_crop(img: Image.Image) -> Image.Image:
    """Applies circular mask to smooth boundaries."""
    w, h = img.size
    min_dim = min(w, h)
    left, top = (w - min_dim) // 2, (h - min_dim) // 2
    img_square = img.crop((left, top, left + min_dim, top + min_dim))
    mask = Image.new("L", (min_dim, min_dim), 0)
    draw = ImageDraw.Draw(mask)
    draw.ellipse((0, 0, min_dim, min_dim), fill=255)
    res = Image.new("RGB", (min_dim, min_dim), (0, 0, 0))
    res.paste(img_square, mask=mask)
    return res

def ben_graham_enhancement(img: Image.Image, sigma: float = 10.0) -> Image.Image:
    """Local average subtraction to remove lighting variations and emphasize microvascular lesions."""
    arr = np.array(img, dtype=np.float32)
    blurred = np.array(img.filter(ImageFilter.GaussianBlur(radius=sigma)), dtype=np.float32)
    enhanced = 4.0 * arr - 4.0 * blurred + 128.0
    enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
    return Image.fromarray(enhanced)

def preprocess_fundus(img: Image.Image, target_size: int = 224, use_ben_graham: bool = False) -> Image.Image:
    img = crop_image_from_gray(img)
    img = circle_crop(img)
    if use_ben_graham:
        img = ben_graham_enhancement(img)
    return img.resize((target_size, target_size), Image.Resampling.LANCZOS)

print("✅ Clinical preprocessing functions compiled successfully!")

In [ ]:
# 5. PyTorch Dataset & Data Augmentations
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, target_size=224, use_ben_graham=False):
        self.img_dir = img_dir
        self.transform = transform
        self.target_size = target_size
        self.use_ben_graham = use_ben_graham
        
        records = []
        for _, row in df.iterrows():
            p = os.path.join(img_dir, f"{row['id_code']}.png")
            if os.path.exists(p):
                records.append((p, int(row['diagnosis']), row['id_code']))
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        p, label, code = self.records[idx]
        with Image.open(p) as raw:
            img = raw.convert("RGB")
        img = preprocess_fundus(img, target_size=self.target_size, use_ben_graham=self.use_ben_graham)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long), code

train_dataset = APTOSDataset(df_train, TRAIN_IMG_DIR, transform=train_transforms)
val_dataset = APTOSDataset(df_val, VAL_IMG_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Calculate class weights for loss function to address class imbalance
total = len(train_dataset)
weights = [total / (5.0 * max(1, (df_train['diagnosis'] == i).sum())) for i in range(5)]
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"Balanced Class Weights: {[round(w, 2) for w in weights]}")

In [ ]:
# 6. Transfer Learning Model Architecture (ResNet Backbone)
def build_model(num_classes=5, backbone="resnet50"):
    if backbone == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for param in list(model.parameters())[:-15]:
            param.requires_grad = False
        num_ftrs = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(256, num_classes),
        )
    else:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for param in list(model.parameters())[:-12]:
            param.requires_grad = False
        num_ftrs = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(num_ftrs, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, num_classes),
        )
    return model

model = build_model(num_classes=5, backbone="resnet50").to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

print("✅ ResNet model initialized with custom clinical classification head!")

In [ ]:
# 7. Full Training & Validation Loop with Quadratic Weighted Kappa (QWK)
EPOCHS = 10
best_kappa = -1.0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_kappa": []}

print(f"🚀 Starting {EPOCHS} Epochs of Deep Learning Training on {device}...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Training Phase ──
    model.train()
    running_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    
    scheduler.step()
    epoch_train_loss = running_loss / len(train_dataset)
    
    # ── Validation Phase ──
    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = accuracy_score(val_targets, val_preds)
    epoch_kappa = cohen_kappa_score(val_targets, val_preds, weights="quadratic")
    
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    history["val_kappa"].append(epoch_kappa)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.1f}% | Quadratic Kappa (κ): {epoch_kappa:.4f}")
    
    # Save best model checkpoint based on Quadratic Weighted Kappa
    if epoch_kappa > best_kappa or epoch == 1:
        best_kappa = epoch_kappa
        torch.save(model.state_dict(), "dr_model.pth")
        print(f"   💾 Checkpoint saved! Best Kappa: {best_kappa:.4f}")

print(f"\n🎉 Training Complete in {(time.time() - start_time) / 60:.2f} minutes! Best Kappa: {best_kappa:.4f}")

In [ ]:
# 8. Confusion Matrix & Clinical Evaluation
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(val_targets, val_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["0: None", "1: Mild", "2: Mod", "3: Sev", "4: PDR"])
disp.plot(cmap="Blues", ax=ax, colorbar=True)
plt.title("Confusion Matrix — Validation Set (APTOS 2019)", fontweight="bold")
plt.show()

print("\nClassification Report:")
print(classification_report(val_targets, val_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# 9. Export Checkpoint & Metadata for Chakshus Platform
metadata = {
    "model_name": "Chakshus AI Diabetic Retinopathy ResNet-50",
    "dataset": "APTOS 2019 Blindness Detection (Aravind Eye Hospital, India)",
    "total_samples": len(df_train) + len(df_val),
    "train_samples": len(df_train),
    "val_samples": len(df_val),
    "best_quadratic_kappa": round(float(best_kappa), 4),
    "final_val_acc": round(float(history['val_acc'][-1]), 4),
    "classes": CLASS_NAMES,
    "weights_file": "dr_model.pth",
    "device": str(device),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Saved 'dr_model.pth' and 'model_metadata.json'!")
print("To use in Chakshus platform: place 'dr_model.pth' inside 'ml-service/models/'")